<a href="https://colab.research.google.com/github/kwabenaafoakwafrempong-cell/STC-Ghana-Predictive-Maintenance-/blob/main/Simulated_STC_Ghana_Synthetic_Dataset_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PART 1 — Import Libraries

import numpy as np
import pandas as pd
from sklearn.utils import shuffle
import os

print("Libraries imported.")

# PART 2 — Generate Synthetic Dataset
# Reproducibility
np.random.seed(42)
n = 1000

print("=" * 70)
print("  SYNTHETIC DATASET GENERATION — STC GHANA FLEET")
print("=" * 70)

# VARIABLE 1 - VEHICLE AGE (years)

vehicle_age = np.random.gamma(shape=3.0, scale=3.0, size=n).clip(1, 25)
print(f"\n  [1/12] vehicle_age_years          — min={vehicle_age.min():.1f},  "
      f"max={vehicle_age.max():.1f},  mean={vehicle_age.mean():.1f}")

# VARIABLE 2 — TOTAL MILEAGE (km)

mileage = (vehicle_age * np.random.normal(70000, 15000, n)).clip(10000, 1500000)
print(f"  [2/12] total_mileage_km           — min={mileage.min():,.0f},  "
      f"max={mileage.max():,.0f},  mean={mileage.mean():,.0f}")

# VARIABLE 3 — ENGINE HOURS (hours)

engine_hours = (mileage / np.random.normal(55, 8, n)).clip(500, 25000)
print(f"  [3/12] engine_hours               — min={engine_hours.min():.0f},  "
      f"max={engine_hours.max():.0f},  mean={engine_hours.mean():.0f}")

# VARIABLE 4 — DAYS SINCE LAST MAINTENANCE

days_since_maintenance = np.random.exponential(scale=45, size=n).clip(1, 365)
print(f"  [4/12] days_since_maintenance     — min={days_since_maintenance.min():.0f},  "
      f"max={days_since_maintenance.max():.0f},  mean={days_since_maintenance.mean():.1f}")

# VARIABLE 5 — NUMBER OF PREVIOUS FAULTS (12 months)

previous_faults = np.random.poisson(lam=2.5, size=n).clip(0, 15)
print(f"  [5/12] previous_faults_12m        — min={previous_faults.min()},  "
      f"max={previous_faults.max()},  mean={previous_faults.mean():.2f}")

# VARIABLE 6 — COMPONENT CONDITION INDEX (0–100, lower = worse)

component_condition = (np.random.beta(a=4, b=2, size=n) * 100).clip(5, 100)
print(f"  [6/12] component_condition_index  — min={component_condition.min():.1f},  "
      f"max={component_condition.max():.1f},  mean={component_condition.mean():.1f}")


# VARIABLE 7 — TYRE CONDITION SCORE (0–10, lower = worse)

tyre_condition = np.random.choice(
    range(0, 11),
    size=n,
    p=[0.02, 0.03, 0.05, 0.08, 0.12, 0.15, 0.18, 0.17, 0.12, 0.06, 0.02]
)
print(f"  [7/12] tyre_condition_score       — min={tyre_condition.min()},  "
      f"max={tyre_condition.max()},  mean={tyre_condition.mean():.2f}")


# VARIABLE 8 — BRAKE SYSTEM CONDITION (0–10, lower = worse)

brake_condition = np.random.choice(
    range(0, 11),
    size=n,
    p=[0.01, 0.02, 0.04, 0.07, 0.11, 0.15, 0.20, 0.20, 0.13, 0.05, 0.02]
)
print(f"  [8/12] brake_condition_score      — min={brake_condition.min()},  "
      f"max={brake_condition.max()},  mean={brake_condition.mean():.2f}")


# VARIABLE 9 — OPERATIONAL INTENSITY (daily km, proxy for load)

operational_intensity = np.random.normal(loc=280, scale=90, size=n).clip(50, 550)
print(f"  [9/12] operational_intensity_kmday— min={operational_intensity.min():.1f},  "
      f"max={operational_intensity.max():.1f},  mean={operational_intensity.mean():.1f}")


# VARIABLE 10 — DRIVER EXPERIENCE (years)

driver_experience = np.random.normal(loc=8, scale=4, size=n).clip(1, 35)
print(f"  [10/12] driver_experience_years   — min={driver_experience.min():.1f},  "
      f"max={driver_experience.max():.1f},  mean={driver_experience.mean():.1f}")


# VARIABLE 11 — ROUTE TYPE (categorical)

route_type = np.random.choice(
    ['Urban', 'Peri-Urban', 'Long-Distance'],
    size=n,
    p=[0.25, 0.35, 0.40]
)
unique_rt, counts_rt = np.unique(route_type, return_counts=True)
print(f"  [11/12] route_type                — "
      + ", ".join([f"{v}:{c}" for v, c in zip(unique_rt, counts_rt)]))


# VARIABLE 12 — LAST MAINTENANCE TYPE (categorical)

maintenance_type = np.random.choice(
    ['Reactive', 'Preventive', 'Condition-Based'],
    size=n,
    p=[0.55, 0.35, 0.10]
)
unique_mt, counts_mt = np.unique(maintenance_type, return_counts=True)
print(f"  [12/12] last_maintenance_type     — "
      + ", ".join([f"{v}:{c}" for v, c in zip(unique_mt, counts_mt)]))



# PART 3 — Construct Binary Target Variable

# TARGET — MAINTENANCE REQUIRED (binary: 1 = required, 0 = not required)
#   vehicle age           - 25%
#   days since maintenance- 20%
#   previous faults       - 20%
#   component condition   - 15%
#   brake condition       - 10%
#   tyre condition        - 10%

risk_score = (
    0.25 * (vehicle_age            / 25)  +
    0.20 * (days_since_maintenance / 365) +
    0.20 * (previous_faults        / 15)  +
    0.15 * (1 - component_condition / 100) +
    0.10 * (1 - brake_condition     / 10)  +
    0.10 * (1 - tyre_condition      / 10)
)

# Controlled Gaussian noise prevents fully deterministic labels and
# introduces realistic label uncertainty (sensor noise, human error)
noise            = np.random.normal(0, 0.05, n)
risk_score_noisy = (risk_score + noise).clip(0, 1)

# 85th-percentile threshold - top 15% flagged as maintenance required
threshold            = np.percentile(risk_score_noisy, 85)
maintenance_required = (risk_score_noisy >= threshold).astype(int)

pos_pct = maintenance_required.mean() * 100
print(f"\n  Threshold applied         : {threshold:.4f}  (85th percentile)")
print(f"  Class 0 (No Maintenance)  : {(maintenance_required==0).sum()}  ({100-pos_pct:.1f}%)")
print(f"  Class 1 (Maint. Required) : {(maintenance_required==1).sum()}  ({pos_pct:.1f}%)")



# PART 4 — Assemble DataFrame & Shuffle

df = pd.DataFrame({
    'vehicle_age_years':           vehicle_age.round(1),
    'total_mileage_km':            mileage.round(0).astype(int),
    'engine_hours':                engine_hours.round(0).astype(int),
    'days_since_maintenance':      days_since_maintenance.round(0).astype(int),
    'previous_faults_12m':         previous_faults,
    'component_condition_index':   component_condition.round(1),
    'tyre_condition_score':        tyre_condition,
    'brake_condition_score':       brake_condition,
    'operational_intensity_kmday': operational_intensity.round(1),
    'driver_experience_years':     driver_experience.round(1),
    'route_type':                  route_type,
    'last_maintenance_type':       maintenance_type,
    'maintenance_required':        maintenance_required
})

# Shuffle to remove any generation-order patterns
df = shuffle(df, random_state=42).reset_index(drop=True)

print(f"\n  DataFrame assembled:  {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Columns: {list(df.columns)}")


# PART 5 — Validation Checks

print("\n" + "=" * 70)
print("  VALIDATION CHECKS")
print("=" * 70)

# Missing values
print(f"\n  Missing values per column:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("     No missing values.")
else:
    print(missing[missing > 0])

# Duplicates
dupes = df.duplicated().sum()
print(f"\n  Duplicate rows : {dupes}  {' ' if dupes == 0 else '  Review required'}")

# Class distribution
print(f"\n  Class distribution (maintenance_required):")
vc = df['maintenance_required'].value_counts().sort_index()
vp = df['maintenance_required'].value_counts(normalize=True).sort_index().round(4)
for cls in [0, 1]:
    label = "No Maintenance" if cls == 0 else "Maintenance Required"
    print(f"    Class {cls} ({label:22s}) : {vc[cls]:4d}  ({vp[cls]*100:.1f}%)")

# Descriptive statistics
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(f"\n  Descriptive statistics (numerical features):")
print(df.describe().round(2).to_string())

# Categorical value counts
print(f"\n  Categorical feature — route_type:")
print(df['route_type'].value_counts().to_string())

print(f"\n  Categorical feature — last_maintenance_type:")
print(df['last_maintenance_type'].value_counts().to_string())

# Range checks
print(f"\n  Range integrity checks:")
checks = {
    'vehicle_age_years'          : (1, 25),
    'total_mileage_km'           : (10000, 1500000),
    'engine_hours'               : (500, 25000),
    'days_since_maintenance'     : (1, 365),
    'previous_faults_12m'        : (0, 15),
    'component_condition_index'  : (5, 100),
    'tyre_condition_score'       : (0, 10),
    'brake_condition_score'      : (0, 10),
    'operational_intensity_kmday': (50, 550),
    'driver_experience_years'    : (1, 35),
}
all_ok = True
for col, (lo, hi) in checks.items():
    ok = (df[col].min() >= lo) and (df[col].max() <= hi)
    status = " " if ok else "  OUT OF RANGE"
    if not ok:
        all_ok = False
    print(f"    {col:<35} [{lo}, {hi}]  →  "
          f"actual [{df[col].min():.1f}, {df[col].max():.1f}]  {status}")
if all_ok:
    print("\n     All range checks passed.")


# PART 7 — Save Dataset to Google Drive

print("\n" + "=" * 70)
print("  SAVING DATASET TO GOOGLE DRIVE")
print("=" * 70)

# Define SAVE_PATH for the CSV file
SAVE_PATH = 'stc_ghana_synthetic_fleet_maintenance.csv'

df.to_csv(SAVE_PATH, index=False)

# Verify the file was written correctly
df_verify      = pd.read_csv(SAVE_PATH)
rows_match     = len(df_verify) == len(df)
cols_match     = list(df_verify.columns) == list(df.columns)
file_size_kb   = os.path.getsize(SAVE_PATH) / 1024

print(f"\n    File saved successfully!")
print(f"\n    File name   : stc_ghana_synthetic_fleet_maintenance.csv")
print(f"    Full path   : {SAVE_PATH}")
print(f"    Dimensions  : {df_verify.shape[0]} rows × {df_verify.shape[1]} columns")
print(f"    File size   : {file_size_kb:.1f} KB")
print(f"  ✔   Rows match  : {rows_match}")
print(f"  ✔   Cols match  : {cols_match}")

Libraries imported.
  SYNTHETIC DATASET GENERATION — STC GHANA FLEET

  [1/12] vehicle_age_years          — min=1.0,  max=25.0,  mean=9.2
  [2/12] total_mileage_km           — min=41,229,  max=1,500,000,  mean=637,189
  [3/12] engine_hours               — min=700,  max=25000,  mean=11645
  [4/12] days_since_maintenance     — min=1,  max=330,  mean=45.1
  [5/12] previous_faults_12m        — min=0,  max=9,  mean=2.46
  [6/12] component_condition_index  — min=12.3,  max=99.1,  mean=66.9
  [7/12] tyre_condition_score       — min=0,  max=10,  mean=5.60
  [8/12] brake_condition_score      — min=0,  max=10,  mean=5.69
  [9/12] operational_intensity_kmday— min=50.0,  max=550.0,  mean=283.5
  [10/12] driver_experience_years   — min=1.0,  max=21.1,  mean=8.2
  [11/12] route_type                — Long-Distance:421, Peri-Urban:346, Urban:233
  [12/12] last_maintenance_type     — Condition-Based:96, Preventive:352, Reactive:552

  Threshold applied         : 0.3731  (85th percentile)
  Class 0 (No 